In [1]:
from binance import Client
from binance.enums import *
from binance.exceptions import BinanceAPIException, BinanceOrderException
import os
import pandas as pd
import time 
import ta
import importlib
import MyOrdersSave
importlib.reload(MyOrdersSave)
from MyOrdersSave import MyOrdersSave as mos
from dotenv import load_dotenv

load_dotenv()

class BotCripto:
    def __init__(self, API_KEY, API_SECRET, symbol, interval, operationCode, qtd):
        connection = Client(API_KEY, API_SECRET)

        self.connection = connection
        self.symbol = symbol
        self.interval = interval
        self.operationCode = operationCode
        self.qtd = qtd
        self.position = self.has_position()
        self.value = self.get_value()
        self.xls_file = 'C:/Users/kaduf/Desktop/BotCripto/orders.xlsx'

    def has_sufficient_balance(self, asset, quantity):
        saldo = self.connection.get_asset_balance(asset=asset)
        disponivel = float(saldo['free'])
        return disponivel >= quantity

    def get_min_quantity(self, operationCode):
        info = self.connection.get_symbol_info(operationCode)
        filter = {f['filterType']: f for f in info['filters']}

        lot_size = filter.get('LOT_SIZE', {})
        min_notonal = filter.get('MIN_NOTIONAL', {}) or filter.get('NOTIONAL', {})

        return {
            'min_quantity': float(lot_size.get('minQty', 0)),
            'step_size': float(lot_size.get('stepSize', 0)),
            'min_notional': float(min_notonal.get('minNotional', 0))
        }

    def has_sufficient_notional(self, qty, price):
        min_notional = self.get_min_quantity(self.operationCode)['min_notional']
        notional_value = qty * price
        if  notional_value < min_notional:
            print(f'\nNotional value {notional_value} is less than the minimum notional {min_notional}')
            return False
        return True

    def has_position(self):
        balance = self.connection.get_asset_balance(asset = self.symbol)
        return float(balance['free']) >= self.qtd

    def get_value(self):
        candle = self.connection.get_klines(symbol = self.operationCode, interval = self.interval, limit=1000)

        value = pd.DataFrame(candle)

        value.columns = ['time', 'open', 'high', 'low', 'close', 
                        'volume', 'close_time', 'quote_asset_volume', 
                        'number_of_trades', 'taker_buy_base_asset_volume', 
                        'taker_buy_quote_asset_volume', 'ignore']

        value = value[['time', 'close']]

        value['close'] = value['close'].astype(float)
        value['time'] = pd.to_datetime(value['time'], unit = 'ms').dt.tz_localize('UTC')
        value['time'] = value['time'].dt.tz_convert('America/Sao_Paulo')

        return value

    def print_order(self, order):
        print('Order details:')
        print(f"Order ID: {order['orderId']}")
        print(f"Symbol: {order['symbol']}")
        print(f"Side: {order['side']}")
        print(f"Type: {order['type']}")
        print(f"Quantity: {order['origQty']}")
        print(f"Price in BRL: {order['cummulativeQuoteQty']}")

    def strategy(self):
        
        self.value['fast_ma'] = self.value['close'].rolling(window = 7).mean()
        self.value['slow_ma'] = self.value['close'].rolling(window = 40).mean()
        self.value['rsi'] = ta.momentum.RSIIndicator(close = self.value['close'], window = 14).rsi()

        last_fast_ma = self.value['fast_ma'].iloc[-1]
        last_slow_ma = self.value['slow_ma'].iloc[-1]
        rsi = self.value['rsi'].iloc[-1]

        print(f'Last Fast MA: {last_fast_ma}\t ||Last Slow MA: {last_slow_ma}\t ||RSI: {rsi}')

        acount = self.connection.get_account()

        for coin in acount['balances']:
            if coin['asset'] == self.symbol:
                actual_qty = float(coin['free'])

        if (last_fast_ma > last_slow_ma and rsi >= 50):
            if self.position == False: 
                self.buy()
                self.position = True
                
        elif (last_fast_ma < last_slow_ma and rsi < 50):
            if self.position == True: 
                self.sell()
                self.position = False                

    def buy(self):
        preco_atual = float(self.connection.get_symbol_ticker(symbol=self.operationCode)['price'])
        if not self.has_sufficient_balance('BRL', self.qtd * preco_atual):
            print(f'\nInsufficient balance to buy {self.qtd} of {self.symbol}\n')
            return None 

        if not self.has_sufficient_notional(self.qtd, preco_atual):
            print(f'\nNotional value of the order is less than the minimum notional required to buy {self.qtd} of {self.symbol}\n')
            return None

        try:
            order = self.connection.create_order(
                symbol = self.operationCode,
                side = SIDE_BUY,
                type = ORDER_TYPE_MARKET,
                quantity = self.qtd,
            )
    
            self.print_order(order)
            print(f'Bought {self.qtd} of {self.symbol}')
            mos(order, self.xls_file).save_to_xls()

            return order

        except BinanceAPIException as e:
            if e.code == -2010:
                print(f'\nInsufficient balance to buy {self.qtd} of {self.symbol}')
            else:
                print(f'\nError occurred while placing buy order: {e}')
            return None
    
    def sell(self):
        if not self.has_sufficient_balance(self.symbol, self.qtd):
            print(f'\nInsufficient balance to sell {self.qtd} of {self.symbol}\n')
            return None

        actual_price = float(self.connection.get_symbol_ticker(symbol=self.operationCode)['price'])
        if not self.has_sufficient_notional(self.qtd, actual_price):
            print(f'\nNotional value of the order is less than the minimum notional required to sell {self.qtd} of {self.symbol}\n')
            return None
        
        try:

            order = self.connection.create_order(
                symbol = self.operationCode,
                side = SIDE_SELL,
                type = ORDER_TYPE_MARKET,
                quantity = self.qtd,
            )

            self.print_order(order)
            print(f'Sold {self.qtd} of {self.symbol}')
            mos(order, self.xls_file).save_to_xls()

            return order

        except BinanceAPIException as e:
            if e.code == -2010:
                print(f'\nInsufficient balance to sell {self.qtd} of {self.symbol}')
            else:
                print(f'\nError occurred while placing sell order: {e}')
            return None

In [ ]:
API_KEY = os.getenv('API_KEY')
API_SECRET = os.getenv('SECRET_KEY')
interval = Client.KLINE_INTERVAL_1MINUTE
operationCode = 'XRPBRL'
symbol = 'XRP'

bot = BotCripto(API_KEY, API_SECRET, symbol, interval, operationCode, 5.0)

minToOperate = bot.get_min_quantity(operationCode)
print(f'Minimum quantity to operate:')
print(f"min_quantity(Ativo): {minToOperate['min_quantity']}")
print(f"step_size (Incrementação permitida): {minToOperate['step_size']}")
print(f"min_notional(BRL): {minToOperate['min_notional']}")
print(f'\nPosition: {bot.position}')

asset_balance = bot.connection.get_asset_balance(asset=symbol)
brl_balance = bot.connection.get_asset_balance(asset='BRL')
print('\nSaldo BRL livre:', brl_balance['free'])
print('Saldo XRP livre:', asset_balance['free'])


Minimum quantity to operate:
min_quantity(Ativo): 0.1
step_size (Incrementação permitida): 0.1
min_notional(BRL): 10.0

Position: False

Saldo BRL livre: 42.43881624
Saldo XRP livre: 0.90000000
Order details:
Order ID: 314269800
Symbol: XRPBRL
Side: BUY
Type: MARKET
Quantity: 5.00000000
Price in BRL: 37.41000000
Bought 5.0 of XRP
Order saved to C:/Users/kaduf/Desktop/BotCripto/orders.xlsx



{'symbol': 'XRPBRL',
 'orderId': 314269800,
 'orderListId': -1,
 'clientOrderId': 'x-HNA2TXFJ88b5f78e870dac05ef1947',
 'transactTime': 1787698981330,
 'price': '0.00000000',
 'origQty': '5.00000000',
 'executedQty': '5.00000000',
 'origQuoteOrderQty': '0.00000000',
 'cummulativeQuoteQty': '37.41000000',
 'status': 'FILLED',
 'timeInForce': 'GTC',
 'type': 'MARKET',
 'side': 'BUY',
 'workingTime': 1787698981330,
 'fills': [{'price': '7.48200000',
   'qty': '5.00000000',
   'commission': '0.00000778',
   'commissionAsset': 'BNB',
   'tradeId': 7023198}],
 'selfTradePreventionMode': 'EXPIRE_MAKER'}

In [ ]:
while True:
    bot.value = bot.get_value()
    print(f'\nAnalize hour: {bot.value["time"].iloc[-1]}')
    print (f'Current Value: {bot.value["close"].iloc[-1]:.8f}')
    bot.strategy()
    time.sleep(60)